In [18]:
from multiprocess import Pool
import itertools
import json
import re
import numpy as np
from datasets import load_dataset

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [ ]:
# !rm -rf malaysian-whole malaysian-segment
!mkdir dialects-whole
!mkdir dialects-segment

In [ ]:
from glob import glob
import os

files = glob('dialects-group/*.json')

In [ ]:
64599-0.mp3

In [ ]:
import copy
import soundfile as sf
import librosa
import numpy as np
from tqdm import tqdm

def loop(files):
    files, _ = files
    combine_all = []
    for f in tqdm(files):
        with open(f) as fopen:
            g = json.load(fopen)

        i = int(os.path.split(f)[1].replace('.json', ''))
        audio_files = []
        timestamps = []
        last_timestamp = 0
        for g_ in g:
            audio_files.append(g_[0]['audio_filename'])
            timestamp = copy.deepcopy(g_[1])
            for k in range(len(timestamp)):
                timestamp[k]['start'] += last_timestamp
                timestamp[k]['end'] += last_timestamp
            timestamps.extend(timestamp)
            last_timestamp = timestamp[-1]['end']
    
        word_level = []
        for t in timestamps:
            start = t['start']
            w = t['text']
            end = t['end']
            word_level.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
        
        segments, temp = [], [timestamps[0]]
        last_t = timestamps[0]['end']
        for c_ in timestamps[1:]:
            if ((c_['start'] - last_t) > 0.4):
                segments.append(temp)
                temp = []
    
            last_t = c_['end']
            temp.append(c_)
    
        if len(temp):
            segments.append(temp)
    
        segment_level = []
        for s in segments:
            start = s[0]['start']
            end = s[-1]['end']
            w = ' '.join([c_['text'] for c_ in s])
            t = f"<|{start:.2f}|> {w}<|{end:.2f}|>"
            segment_level.append(t)
    
        y = [librosa.load(f, sr = 16000)[0] for f in audio_files]
        y = np.concatenate(y)
    
        audio_filename = f'dialects-whole/{i}.mp3'
        sf.write(audio_filename, y, 16000)
    
        segment_audio_filenames = []
        streaming_word_level = []
        for k, s in enumerate(segments):
            segment_audio_filename = f'dialects-segment/{i}-{k}.mp3'
            start = s[0]['start']
            end = s[-1]['end']
            y_ = y[int(start * 16000): int(end * 16000)]
            sf.write(segment_audio_filename, y_, 16000)
            segment_audio_filenames.append(segment_audio_filename)

            del y_
    
            word_level_ = []
            for t in s:
                start = t['start']
                w = t['text']
                end = t['end']
                word_level_.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
            streaming_word_level.append(''.join(word_level_))
            
        word_level = ''.join(word_level)
    
        combine_all.append({
            'mode': 'whole',
            'level': 'segment',
            'texts': [''.join(segment_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'whole',
            'level': 'word',
            'texts': [''.join(word_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'segment',
            'texts': segment_level,
            'audio_filenames': segment_audio_filenames,
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'word',
            'texts': streaming_word_level,
            'audio_filenames': segment_audio_filenames,
        })

        del y, timestamps, segments, segment_level, word_level, streaming_word_level
        
    return combine_all

In [ ]:
combine_all = loop((files[:10], 0))

In [ ]:
combine_all = multiprocessing(files, loop, cores = 30)

In [1]:
import IPython.display as ipd
ipd.Audio('dialects-segment/883755-0.mp3')

In [4]:
!ls -lh parlimen-segment/11050-5.mp3

-rw-r--r-- 1 ubuntu ubuntu 0 Jul 16 00:59 parlimen-segment/11050-5.mp3


In [6]:
from glob import glob

files = glob('done-dialects/*.json')
len(files)

998143

In [19]:
with open('done-dialects/64599.json') as fopen:
    d = json.load(fopen)

d

[{'mode': 'whole',
  'level': 'segment',
  'texts': ['<|0.10|> B,<|0.10|><|1.32|> supaya tidak<|1.90|><|3.40|> kalau dibuang, dia akan menjadi makanan kepada<|5.76|><|7.08|> syaitan. Jadi dua sebab.<|8.34|><|8.92|> Kenapa makanan<|9.80|><|10.52|> perlu untuk dihabiskan.<|11.64|><|12.78|> Termasuk yang ada di jari, termasuk yang ada di pinggan, termasuk yang jatuh.<|15.74|><|16.34|> Yang<|16.48|><|16.94|> pertama,<|17.28|><|17.70|> kerana kita<|18.12|><|18.60|> tak tahu di mana keberkatan terletak pada makanan. Mungkin kepada yang jatuh itu.<|21.98|><|22.60|> Yang kedua,<|22.96|><|23.38|> untuk tidak menjadi makanan itu, menjadi makanan kepada syaitan, yang syaitan akan mengganggu kepada<|28.06|><|29.40|> kita.<|29.98|><|31.18|> Kecuali bila mana dah<|32.54|><|32.98|> apa namanya, makanan itu tak boleh lagi dimakan dah.<|35.48|><|38.10|> Bila mana boleh untuk dihantarkan kepada<|40.72|><|42.46|> apa namanya, tempat-tempat yang boleh<|44.40|><|45.24|> layak untuk menerima. Tapi janganlah

In [9]:
import json
from tqdm import tqdm

data = []
for f in tqdm(files):
    with open(f) as fopen:
        d = json.load(fopen)
    data.extend(d)

len(d)

100%|██████████| 998143/998143 [07:08<00:00, 2330.79it/s]


4

In [11]:
len(data)

3992572

In [14]:
from datasets import Dataset

dataset = Dataset.from_list(data)

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
dataset.push_to_hub('malaysia-ai/Malaysian-STT', 'dialects')

Uploading the dataset shards: 100%|██████████| 18/18 [01:11<00:00,  3.95s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Malaysian-STT/commit/6b90eeafb90d23293c14283a38a503bf6190b281', commit_message='Upload dataset', commit_description='', oid='6b90eeafb90d23293c14283a38a503bf6190b281', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Malaysian-STT', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Malaysian-STT'), pr_revision=None, pr_num=None)